<a href="https://colab.research.google.com/github/Su-3214/26SD424_MASUZAKI_TATSUYA/blob/main/AI%E6%BC%94%E7%BF%9220%E6%9C%80%E7%B5%82%E8%AA%B2%E9%A1%8C%E3%83%AF%E3%83%BC%E3%82%AF%E3%83%95%E3%83%AD%E3%83%BC%E3%82%A2%E3%83%97%E3%83%AA%E3%82%B1%E3%83%BC%E3%82%B7%E3%83%A7%E3%83%B3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q langchain langchain-google-genai langchain-community tavily-python==0.5.0 langgraph

In [4]:
import os
import time
from google.colab import userdata
from langchain_community.retrievers import TavilySearchAPIRetriever
from langchain_google_genai import ChatGoogleGenerativeAI

# APIキーの設定
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

# モデルと検索ツールの初期化 (指定のgemini-flash-latestを使用)
model = ChatGoogleGenerativeAI(model="gemini-flash-latest", google_api_key=GEMINI_API_KEY, temperature=0.5)
retriever = TavilySearchAPIRetriever(k=4)

# エラー発生時のみ指数バックオフで自動再試行する高度なラッパー関数
def safe_invoke(prompt_text, max_retries=5):
    """
    通常時は待ち時間なしで実行。
    API制限(429)やサーバー障害(503)が発生した場合のみ、待機時間を倍増させながら再試行する。
    """
    for attempt in range(max_retries):
        try:
            return model.invoke(prompt_text)
        except Exception as e:
            wait_time = (2 ** attempt) * 10 + 5
            print(f"    ⚠️ APIエラーを検出しました ({e})")
            print(f"    {wait_time}秒待機してから再試行します (試行 {attempt + 1}/{max_retries})...")
            time.sleep(wait_time)

    raise Exception("❌ API呼び出しの最大再試行回数に達しました。時間をおいて再実行してください。")

print("環境セットアップおよび堅牢化通信ロジックの実装が完了しました。")

環境セットアップおよび堅牢化通信ロジックの実装が完了しました。


In [5]:
from typing import TypedDict

class State(TypedDict):
    query: str          # ユーザーからの検索テーマ
    plan: str           # 記事構成と検索キーワード
    context: str        # 取得したWeb情報
    draft: str          # 草稿
    fact_ok: bool       # 事実確認の判定
    answer: str         # 最終記事
    is_good: bool       # 最終品質チェック結果
    feedback: str       # 修正指示
    retry_count: int    # 書き直しの試行回数

# ① 計画エージェント
def plan_node(state: State):
    print("--- [1/6] 計画エージェントが構成と検索方針を策定中 ---")
    prompt = f"テーマ「{state['query']}」について記事を書くための構成案（3つの見出し）と、検索エンジンで調べるべきキーワードを3つ提案してください。"
    response = safe_invoke(prompt)
    return {"plan": response.content}

# ② 検索エージェント（TPM制限対策としてテキスト長を安全サイズに自動カット）
def search_node(state: State):
    print("--- [2/6] 検索エージェントがWebから情報を収集中 ---")
    docs = retriever.invoke(state["query"])
    context = "\n".join([doc.page_content for doc in docs])

    # トークン数上限（TPM制限）を超えないよう最大文字数を制限（約2000文字まで）
    MAX_CONTEXT_CHARS = 2000
    if len(context) > MAX_CONTEXT_CHARS:
        context = context[:MAX_CONTEXT_CHARS] + "\n...(以下略)"
        print(f"    ℹ️ 検索テキストが長いため、安全のため先頭{MAX_CONTEXT_CHARS}文字に切り詰めました。")

    return {"context": context, "retry_count": state.get("retry_count", 0)}

In [6]:
# ③ 執筆エージェント
def write_node(state: State):
    print("--- [3/6] 執筆エージェントが草稿を作成中 ---")
    feedback_text = f"\n前回の指摘事項: {state.get('feedback', '')}\n上記を修正して書き直してください。" if state.get('feedback') else ""
    prompt = f"以下の【構成案】と【検索情報】をもとに、「{state['query']}」に関する記事の草稿を作成してください。\n【構成案】\n{state['plan']}\n【検索情報】\n{state['context']}\n{feedback_text}"
    response = safe_invoke(prompt)
    return {"draft": response.content}

# ④ 事実確認エージェント
def fact_check_node(state: State):
    print("--- [4/6] 事実確認エージェントが内容を精査中 ---")
    prompt = f"以下の【検索情報】のみを事実として、【草稿】に矛盾や事実無根の記述がないかチェックしてください。\n【検索情報】\n{state['context']}\n【草稿】\n{state['draft']}\n問題なければ「【判定】問題なし」、矛盾があれば「【判定】矛盾あり」とし、理由を添えてください。"
    response = safe_invoke(prompt).content
    print(f"    評価結果: {response[:30]}...")

    if "【判定】問題なし" in response:
        return {"fact_ok": True, "feedback": ""}
    else:
        return {"fact_ok": False, "feedback": response}

# ⑤ 編集エージェント
def edit_node(state: State):
    print("--- [5/6] 編集エージェントが記事を装飾中 ---")
    prompt = f"以下の【草稿】を、Webメディアの記事として魅力的に読めるようにMarkdown形式で編集・装飾してください。\n【草稿】\n{state['draft']}"
    response = safe_invoke(prompt)
    return {"answer": response.content}

# ⑥ 評価エージェント
def check_node(state: State):
    print("--- [6/6] 評価エージェントが最終品質をチェック中 ---")
    prompt = f"以下の記事が、一般読者にとって読みやすく、適切な文字数（短すぎないか）であるか評価してください。\n【記事内容】\n{state['answer']}\n要件を満たせば「【判定】OK」、修正が必要なら「【判定】NG」とし、理由を記載してください。"
    response = safe_invoke(prompt).content
    print(f"    最終評価: {response[:30]}...")

    if "【判定】OK" in response:
        return {"is_good": True, "feedback": ""}
    else:
        return {"is_good": False, "feedback": response, "retry_count": state["retry_count"] + 1}

In [7]:
# 条件分岐1：ファクトチェック後の判断関数
def decide_after_fact_check(state: State):
    if state["fact_ok"]:
        return "Editor"
    else:
        print(">> ⚠️ 事実確認NG。執筆エージェントへ差し戻します。")
        return "Writer"

# 条件分岐2：最終評価後の判断関数（上限：3回まで再試行可能）
def decide_final(state: State):
    if state["is_good"]:
        print(">> ✅ 最終評価クリア。記事を完成とします。")
        return END
    elif state["retry_count"] >= 3:
        print(">> ⚠️ 再試行の上限（3回）に達しました。現在の状態を出力します（無料枠保護）。")
        return END
    else:
        print(f">> ⚠️ 最終評価NG（再試行 {state['retry_count']}/3 回目）。執筆エージェントへ差し戻します。")
        return "Writer"

In [8]:
from langgraph.graph import StateGraph, END

class TrendCatcherApp:
    def __init__(self):
        """【準備フェーズ】グラフの構造を組み立ててコンパイルする"""
        workflow = StateGraph(State)

        # 1. ノードの登録（外部のノード関数を結びつける）
        workflow.add_node("Planner", plan_node)
        workflow.add_node("Searcher", search_node)
        workflow.add_node("Writer", write_node)
        workflow.add_node("FactChecker", fact_check_node)
        workflow.add_node("Editor", edit_node)
        workflow.add_node("Evaluator", check_node)

        # 2. 処理順序（エッジ）の設定
        workflow.set_entry_point("Planner")
        workflow.add_edge("Planner", "Searcher")
        workflow.add_edge("Searcher", "Writer")
        workflow.add_edge("Writer", "FactChecker")

        # 3. 条件付きエッジの設定（外部の条件分岐関数を指定）
        workflow.add_conditional_edges("FactChecker", decide_after_fact_check)
        workflow.add_edge("Editor", "Evaluator")
        workflow.add_conditional_edges("Evaluator", decide_final)

        # 4. グラフのコンパイル
        self.app = workflow.compile()
        print("🔧 『Trend-Catcher AI』システムの初期化が完了しました！")

    def run(self):
        """【実行フェーズ】ユーザー入力を受け取りワークフローを稼働させる"""
        print("\n==================================================")
        print(" 🤖 『Trend-Catcher AI』へようこそ！")
        print("==================================================")

        # Colabの画面からテーマの入力を受け付ける
        target_query = input("📝 記事にしてほしい最新トレンドのテーマを入力してください：\n> ")

        if not target_query:
            print("テーマが入力されなかったため終了します。")
            return

        print(f"\n【テーマ】『{target_query}』の調査と執筆を開始します。")

        initial_state = {
            "query": target_query,
            "retry_count": 0
        }

        # コンパイル済みのグラフを実行（上記の各ノード関数が順に呼ばれます）
        start_time = time.time()
        final_state = self.app.invoke(initial_state)
        elapsed_time = time.time() - start_time

        print("\n==================================================")
        print(f" ✨ 完成したトレンド記事：『{target_query}』 (処理時間: {int(elapsed_time)}秒)")
        print("==================================================\n")
        print(final_state["answer"])
        print("\n==================================================")

In [9]:
# インスタンス生成（__init__が呼ばれてグラフが構築されます）
my_app = TrendCatcherApp()

# 1回目の実行（例：「2026年 自動運転車の最新動向」）
my_app.run()

🔧 『Trend-Catcher AI』システムの初期化が完了しました！

 🤖 『Trend-Catcher AI』へようこそ！
📝 記事にしてほしい最新トレンドのテーマを入力してください：
> 人類はなぜ争うのか

【テーマ】『人類はなぜ争うのか』の調査と執筆を開始します。
--- [1/6] 計画エージェントが構成と検索方針を策定中 ---
--- [2/6] 検索エージェントがWebから情報を収集中 ---
--- [3/6] 執筆エージェントが草稿を作成中 ---
--- [4/6] 事実確認エージェントが内容を精査中 ---
    評価結果: [{'type': 'text', 'text': '【判定】矛盾あり\n\n【理由】\n検索情報に記載されていない（事実として確認できない）記述が草稿に多数含まれています。\n\n1. **「安全保障のジレンマ」に関する記述**\n草稿の「見出し3」において、軍備増強の悪循環を生む「安全保障のジレンマ」について詳しく解説されていますが、検索情報にはこの用語や具体的な仕組みについての記述が一切ありません。\n\n2. **争いを防ぐための具体的な提言・解決策**\n草稿では「『内集団』の概念を広げる」「対話と相互理解の維持」といった具体的な解決策が提示されていますが、検索情報にはこのような提言や解決策に関する内容は記載されていません。\n\n3. **狩猟採集・農耕牧畜に関する詳細な記述**\n草稿では「食料、安全な住処、配偶者」といった具体的な資源の奪い合いや、農耕牧畜による「持てる者と持たざる者」という概念の発生について言及されています。しかし、検索情報にあるのは「狩猟採取から農耕牧畜へ」というキーワードのみで、それらの詳細な背景や具体例は含まれていません。', 'extras': {'signature': 'EphDCpVDARFNMg/F9O/s5+1AnVTfnxctzu6oZ2SLusRhaZ3OlD8vN56y3uziJUQUumt3qW6shFPtfytGtTBu08gRubI4jP31Nhqi7eU91iMn0Cm2P9tFg6l0VTgCTx2MZcNRUl3xnmfTKBnYteHA8qNaEqhI8WMFWlsP3FHZTQ7qVrU58WlcT3c3B4k7zZiD7LruNQEX+2

Exception: ❌ API呼び出しの最大再試行回数に達しました。時間をおいて再実行してください。